In [2]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
import pyarrow.parquet as pq
import sentencepiece as spm
torch.manual_seed(42)



pf = pq.ParquetFile('data/tinystories0.parquet')


In [3]:
with open('data/input.txt', 'w', encoding='utf-8') as f:
    for i in range(100):
        text_list = pf.read_row_group(i).column('text').to_pylist() # 1000 rows each row group     
        for story in text_list:
            f.write(story + "<|E|>\n")
# currently written: groups 1-10

In [4]:
spm.SentencePieceTrainer.train( # pyright: ignore[reportAttributeAccessIssue]
    input = 'data/input.txt',
    model_prefix = 'token_model',
    vocab_size = 4096,
    model_type = 'bpe',
    user_defined_symbols = ['<|E|>'],
    character_coverage = 1.0,
    byte_fallback = True,
    add_dummy_prefix = True
    )

In [ ]:
sp = spm.SentencePieceProcessor(model_file="models/token_model.model") # pyright: ignore[reportCallIssue]
sp.Encode("Lily found a needle.<|E|>") # pyright: ignore[reportAttributeAccessIssue]

[351, 590, 262, 3915, 4008, 3]

In [5]:
text = open('data/input.txt', 'r').read()

In [10]:
def encode(text):
    return sp.Encode(text)

def decode(list):
    return sp.Decode(list)

In [15]:
encode("he saw something shiny! It was a golden penny!")

[283, 413, 556, 965, 4026, 409, 284, 262, 2531, 299, 3475, 4026]

In [6]:
config = mn.Config()

In [7]:

config.n_embd = 256
config.context_length = 256
config.n_head = 8
config.n_layer = 8
config.batch_size = 64
config.device = 'cuda'
max_iter = 6000
config.vocab_size = 4096

In [10]:
data = torch.tensor(encode(text), dtype=torch.long, device=config.device)

n = int(0.9 * len(data))
train = data[:n]
val = data[n:]

def get_batch(split):
    x = train if split == 'train' else val
    ix = torch.randint(0, len(x) - config.context_length, (config.batch_size,), device=config.device)
    xs = torch.stack([x[i:i+config.context_length] for i in ix]) # (B, T)
    ys = torch.stack([x[i+1:i+config.context_length+1] for i in ix]) # (B, T)
    return xs, ys

In [8]:
model = mn.Transformer(config, dtype=torch.float32, device=config.device)


In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4)

In [ ]:
# # Load the list of tensors
# loaded_params = torch.load("models/params_new.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

C:\Users\Mrp1G\AppData\Local\Temp\ipykernel_18928\202862220.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_params = torch.load("params_new.pth")


In [21]:
for i in range(max_iter):
    t_start = time.time()
    # forward pass
    xs, ys = get_batch('train')
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        logits, loss = model(xs, targets=ys)
    
    # backward pass
    for p in model.parameters():
        p.grad = None
    loss.backward()
    
    # update
    optimizer.step()
    if i % 25 == 0:
        torch.cuda.synchronize()
        t_end = time.time()
        print(f"Step {i}: Loss = {loss.item()} | {(t_end - t_start)*1000.0:.3f}ms")

Step 0: Loss = 1.924696922302246 | 6767.185ms
Step 25: Loss = 1.8618314266204834 | 3850.764ms
Step 50: Loss = 1.8837186098098755 | 3934.664ms
Step 75: Loss = 1.8112061023712158 | 5035.365ms


KeyboardInterrupt: 

In [ ]:
params_list = list(model.parameters())
torch.save(params_list, "models/params_new.pth")

In [27]:
xs, ys = get_batch('val')

logits, loss = model.forward(xs, ys)
loss

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 8.00 GiB of which 0 bytes is free. Of the allocated memory 14.20 GiB is allocated by PyTorch, and 123.68 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [13]:
print(decode(model.generate(torch.tensor([[3]], dtype=torch.long, device=config.device), 2000)[0].tolist()))

<|E|> One day, a boy named Terry and his mom had a great idea. They decided to go to the restaurant maker. Terry was ordinary. He was excited to go shopping through the computers and pay to his destination. When they arrived, Terry was so excited! He quickly grabbed a handful of them and took one. He watched as the orderer crer fixed his ear and flew off to the top with the pictures. It was the perfect way to stay. Everyone was so happy as the food was sent to that paste! They danced and played together all day. When it was time to go home, Terry didn't want to leave. Suddenly, he heard an urge of farm. He hopped away to find another place in the big field! Terry was so happy, he jumped up and gave the people a big welcome. Everyone was so surprised to see the sandwiches and the other chickens home in its mind. Everyone was very happy, and they were never envious of the new toys!<|E|> Bob went to visit the chubby creeker. He was very excited. The creeker showed him lots of interesting 